# Rotations, Orientations, And Batch Primitives

PyTex distinguishes between a mathematical rotation and an orientation that lives
between a crystal frame and a specimen frame. This notebook also shows how the batch
primitives let large vectorized workflows stay semantically typed.

## Key Mathematical Ideas

A unit quaternion is stored in `w, x, y, z` order and defines the active rotation

$$
\mathbf{v}' = \mathbf{R}(q)\,\mathbf{v}.
$$

An `Orientation` then wraps that rotation together with the frame pair
`(crystal_frame, specimen_frame)`.


In [ ]:
import numpy as np

from pytex import (
    AcquisitionGeometry,
    BenchmarkManifest,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    ReferenceFrame,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, *_ = make_context()

eulers = np.array(
    [
        [0.0, 0.0, 0.0],
        [45.0, 35.0, 15.0],
        [90.0, 0.0, 0.0],
    ]
)

euler_set = EulerSet(eulers, convention="bunge", degrees=True)
rotation_set = euler_set.to_rotation_set()
quaternion_set = rotation_set.as_quaternion_set()
orientation_set = OrientationSet.from_euler_angles(
    euler_set,
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=crystal),
)

print("Euler angles")
print(euler_set.as_array())
print("Quaternions")
print(quaternion_set.as_array())


In [ ]:
crystal_vectors = VectorSet(
    values=np.array(
        [
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
        ]
    ),
    reference_frame=crystal,
)

specimen_vectors = orientation_set.map_crystal_directions(crystal_vectors)
print(specimen_vectors.reference_frame.name)
print(specimen_vectors.values)


## Batch Semantics

The important point is not just that these operations are vectorized. It is that the
shared convention or frame survives the operation:

- `EulerSet` retains the Euler convention
- `QuaternionSet` retains canonical quaternion normalization
- `RotationSet` rotates many vectors at once
- `OrientationSet` carries crystal/specimen meaning while doing batched direction maps
